# NEU-DET Steel Surface Defect Detection 

#  0. Installations

In [1]:

import subprocess, sys, os

subprocess.run([sys.executable, '-m', 'pip', 'install', 'albumentations', '-q'], check=False)

import math, copy, random
import numpy as np
import pandas as pd
import torch
import torchvision
import xml.etree.ElementTree as ET
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
import torchvision.transforms.functional as F
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone
from torchvision.models.detection.faster_rcnn import FasterRCNN, FastRCNNPredictor
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import nms
import torch.nn.functional as torch_F
from collections import Counter

os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

if torch.cuda.is_available():
    torch.backends.cudnn.enabled   = False
    torch.backends.cudnn.benchmark = False
    DEVICE = torch.device('cuda')
    name   = torch.cuda.get_device_name(0)
    cap    = torch.cuda.get_device_capability(0)
    vram   = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    print(f'GPU : {name}  |  compute cap: {cap}  |  VRAM: {vram} GB')
    print(f'torch: {torch.__version__}  |  torchvision: {torchvision.__version__}')
    print('cudnn DISABLED — using generic CUDA kernels (fixes kernel image error)')
else:
    DEVICE = torch.device('cpu')
    print('No GPU — running on CPU (slow but works)')

for dirname, _, filenames in os.walk('/kaggle/input'):
    for f in filenames[:15]:
        print(os.path.join(dirname, f))

GPU : Tesla T4  |  compute cap: (7, 5)  |  VRAM: 15.6 GB
torch: 2.10.0+cu128  |  torchvision: 0.25.0+cu128
cudnn DISABLED — using generic CUDA kernels (fixes kernel image error)
/kaggle/input/competitions/teep-internship-program-hucenrotia-lab/submission.csv
/kaggle/input/competitions/teep-internship-program-hucenrotia-lab/Dataset/train_images/pitted_surface_gay7i73fhffn.jpg
/kaggle/input/competitions/teep-internship-program-hucenrotia-lab/Dataset/train_images/patches_1zv1k18lbav2.jpg
/kaggle/input/competitions/teep-internship-program-hucenrotia-lab/Dataset/train_images/scratches_rcfzhlihrdau.jpg
/kaggle/input/competitions/teep-internship-program-hucenrotia-lab/Dataset/train_images/scratches_fvmm5t1evw7y.jpg
/kaggle/input/competitions/teep-internship-program-hucenrotia-lab/Dataset/train_images/scratches_pcgal0tq7lw4.jpg
/kaggle/input/competitions/teep-internship-program-hucenrotia-lab/Dataset/train_images/patches_i31v5waomc1h.jpg
/kaggle/input/competitions/teep-internship-program-hucen

# 1. Configuration

In [2]:

DATASET_ROOT   = '/kaggle/input/competitions/teep-internship-program-hucenrotia-lab/Dataset'
TRAIN_IMG_ROOT = DATASET_ROOT + '/train_images'
TRAIN_ANN_ROOT = DATASET_ROOT + '/train_annotations'
VAL_IMG_ROOT   = '/kaggle/input/competitions/teep-internship-program-hucenrotia-lab/Dataset/test'

NUM_CLASSES    = 7      
EPOCHS         =1   
BATCH_SIZE     = 6      
LEARNING_RATE  = 2e-4   
WEIGHT_DECAY   = 1e-4
CONF_THRESHOLD = 0.35
NMS_THRESHOLD  = 0.45
USE_TTA        = True
INFER_ONLY     = False  # True = skip training, load saved checkpoint
IMG_SIZE       = 640
SEED           = 42

OUTPUT_DIR      = '/kaggle/working'
SUBMISSION_PATH = OUTPUT_DIR + '/submission.csv'
CHECKPOINT_PATH = OUTPUT_DIR + '/checkpoint_best.pth'
EMA_PATH        = OUTPUT_DIR + '/ema_model.pth'
VIZ_DIR         = OUTPUT_DIR + '/output_viz'
os.makedirs(VIZ_DIR, exist_ok=True)

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

LABEL_MAP = {
    'crazing': 1, 'inclusion': 2, 'patches': 3,
    'pitted_surface': 4, 'rolled-in_scale': 5, 'scratches': 6,
}
REV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}
CLASS_NAMES   = ['bg'] + list(LABEL_MAP.keys())

for name, path in [('train_images', TRAIN_IMG_ROOT),
                   ('annotations',  TRAIN_ANN_ROOT),
                   ('test',         VAL_IMG_ROOT)]:
    ok = os.path.exists(path)
    n  = len(os.listdir(path)) if ok else 0
    print(f"{'OK' if ok else 'MISSING':6s}  {name:20s}  ({n} files)  {path}")

OK      train_images          (1440 files)  /kaggle/input/competitions/teep-internship-program-hucenrotia-lab/Dataset/train_images
OK      annotations           (1440 files)  /kaggle/input/competitions/teep-internship-program-hucenrotia-lab/Dataset/train_annotations
OK      test                  (360 files)  /kaggle/input/competitions/teep-internship-program-hucenrotia-lab/Dataset/test


# 2. Augmentations 

In [3]:

def get_train_transforms(img_size=IMG_SIZE):
    return A.Compose([
        A.LongestMaxSize(max_size=img_size),
        A.PadIfNeeded(img_size, img_size, border_mode=0, value=0),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.RandomRotate90(p=0.3),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.2,
                           rotate_limit=15, p=0.5, border_mode=0),
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
        A.GaussNoise(var_limit=(10, 50), p=0.4),
        A.GaussianBlur(blur_limit=(3, 5), p=0.3),
        A.CLAHE(clip_limit=4.0, p=0.4),
        A.Sharpen(alpha=(0.2, 0.5), p=0.3),
        A.CoarseDropout(max_holes=6, max_height=32, max_width=32, fill_value=0, p=0.3),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels'],
                                min_area=16, min_visibility=0.3))

def get_val_transforms(img_size=IMG_SIZE):
    return A.Compose([
        A.LongestMaxSize(max_size=img_size),
        A.PadIfNeeded(img_size, img_size, border_mode=0, value=0),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels'],
                                min_area=1, min_visibility=0.1))

print('Augmentations ready')

Augmentations ready


# 3. Dataset 

In [4]:

class NEUDETDataset(Dataset):
    def __init__(self, img_dir, ann_dir, transforms=None):
        self.img_dir    = img_dir
        self.ann_dir    = ann_dir
        self.transforms = transforms
        self.imgs = sorted([f for f in os.listdir(img_dir) if f.lower().endswith('.jpg')])
        self.all_labels = []
        for fname in self.imgs:
            ann  = os.path.join(ann_dir, fname.replace('.jpg', '.xml'))
            lbls = [LABEL_MAP[obj.find('name').text.strip()]
                    for obj in ET.parse(ann).getroot().findall('object')]
            self.all_labels.append(lbls[0] if lbls else 0)

    def __getitem__(self, idx):
        img_name = self.imgs[idx]
        image    = np.array(Image.open(os.path.join(self.img_dir, img_name)).convert('RGB'))
        ann_path = os.path.join(self.ann_dir, img_name.replace('.jpg', '.xml'))

        boxes, labels = [], []
        for obj in ET.parse(ann_path).getroot().findall('object'):
            lbl  = obj.find('name').text.strip()
            bbox = obj.find('bndbox')
            xmin = int(float(bbox.find('xmin').text))
            ymin = int(float(bbox.find('ymin').text))
            xmax = int(float(bbox.find('xmax').text))
            ymax = int(float(bbox.find('ymax').text))
            h, w = image.shape[:2]
            xmin, xmax = max(0, xmin), min(w, xmax)
            ymin, ymax = max(0, ymin), min(h, ymax)
            if xmax > xmin + 2 and ymax > ymin + 2:
                boxes.append([xmin, ymin, xmax, ymax])
                labels.append(LABEL_MAP[lbl])

        if self.transforms and boxes:
            try:
                aug    = self.transforms(image=image, bboxes=boxes, class_labels=labels)
                image  = aug['image']
                boxes  = list(aug['bboxes'])
                labels = list(aug['class_labels'])
            except Exception:
                image = torch.from_numpy(image.transpose(2, 0, 1)).float() / 255.0
        elif self.transforms:
            aug   = self.transforms(image=image, bboxes=[], class_labels=[])
            image = aug['image']
        else:
            image = torch.from_numpy(image.transpose(2, 0, 1)).float() / 255.0

        boxes  = (torch.as_tensor(boxes,  dtype=torch.float32)
                  if boxes else torch.zeros((0, 4), dtype=torch.float32))
        labels = (torch.as_tensor(labels, dtype=torch.int64)
                  if labels else torch.zeros((0,),  dtype=torch.int64))
        area   = ((boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
                  if len(boxes) else torch.zeros(0, dtype=torch.float32))
        target = {
            'boxes':    boxes,
            'labels':   labels,
            'image_id': torch.tensor([idx], dtype=torch.int64),
            'area':     area,
            'iscrowd':  torch.zeros(len(labels), dtype=torch.int64),
        }
        return image, target

    def __len__(self):
        return len(self.imgs)


def get_balanced_sampler(dataset):
    label_counts   = np.maximum(np.bincount(dataset.all_labels, minlength=NUM_CLASSES), 1)
    weights        = 1.0 / label_counts
    sample_weights = torch.tensor([weights[l] for l in dataset.all_labels], dtype=torch.float)
    return WeightedRandomSampler(sample_weights, len(sample_weights))

def collate_fn(batch):
    return tuple(zip(*batch))

_ds = NEUDETDataset(TRAIN_IMG_ROOT, TRAIN_ANN_ROOT)
print(f'Train samples: {len(_ds)}')
dist = Counter(_ds.all_labels)
print('Class distribution:')
for cls_id, count in sorted(dist.items()):
    print(f'  {REV_LABEL_MAP.get(cls_id, "background"):20s}: {count}')

Train samples: 1440
Class distribution:
  crazing             : 240
  inclusion           : 240
  patches             : 240
  pitted_surface      : 240
  rolled-in_scale     : 240
  scratches           : 240


# 4. Model 

In [ ]:

def get_model(num_classes=NUM_CLASSES):
    try:
        backbone = resnet_fpn_backbone(
            backbone_name='resnet101',
            weights='ResNet101_Weights.IMAGENET1K_V1',
            trainable_layers=5)
        anchor_gen = AnchorGenerator(
            sizes=((16,), (32,), (64,), (128,), (256,)),
            aspect_ratios=((0.25, 0.5, 1.0, 2.0, 4.0),) * 5)
        model = FasterRCNN(
            backbone, num_classes=num_classes,
            rpn_anchor_generator=anchor_gen,
            box_detections_per_img=200,
            rpn_nms_thresh=0.7,
            box_nms_thresh=NMS_THRESHOLD,
            box_score_thresh=CONF_THRESHOLD,
            rpn_pre_nms_top_n_train=4000,  rpn_post_nms_top_n_train=2000,
            rpn_pre_nms_top_n_test=2000,   rpn_post_nms_top_n_test=1000)
        print('Model: Faster R-CNN ResNet-101 FPN (pretrained ImageNet)')
    except Exception as e:
        print(f'ResNet-101 failed ({e.__class__.__name__}), using ResNet-50 COCO...')
        model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
        in_feat = model.roi_heads.box_predictor.cls_score.in_features
        model.roi_heads.box_predictor = FastRCNNPredictor(in_feat, num_classes)
        anchor_gen = AnchorGenerator(
            sizes=((16,), (32,), (64,), (128,), (256,)),
            aspect_ratios=((0.25, 0.5, 1.0, 2.0, 4.0),) * 5)
        model.rpn.anchor_generator         = anchor_gen
        model.rpn.nms_thresh               = 0.7
        model.roi_heads.nms_thresh         = NMS_THRESHOLD
        model.roi_heads.score_thresh       = CONF_THRESHOLD
        model.roi_heads.detections_per_img = 200
        print('Model: Faster R-CNN ResNet-50 FPN (pretrained COCO)')
    return model.to(DEVICE)

model = get_model()
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

#  5. EMA + scheduler + checkpoint 

In [ ]:

class ModelEMA:
    def __init__(self, model, decay=0.9999):
        self.ema   = copy.deepcopy(model).eval()
        self.decay = decay
        for p in self.ema.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        for ep, mp in zip(self.ema.parameters(), model.parameters()):
            ep.data.mul_(self.decay).add_(mp.data, alpha=1.0 - self.decay)


def warmup_cosine_schedule(optimizer, warmup_epochs, total_epochs, steps_per_epoch):
    warmup_steps = warmup_epochs * steps_per_epoch
    total_steps  = total_epochs  * steps_per_epoch
    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def save_checkpoint(state, path):
    torch.save(state, path)

print('EMA / scheduler / checkpoint helpers ready')

#  6. mAP evaluation 

In [ ]:

def compute_map(model, val_loader, iou_threshold=0.5):
    model.eval()
    all_preds  = {i: [] for i in range(1, NUM_CLASSES)}
    all_gts    = {i: [] for i in range(1, NUM_CLASSES)}
    img_offset = 0
    with torch.no_grad():
        for images, targets in tqdm(val_loader, desc='Evaluating', leave=False):
            images  = [img.to(DEVICE) for img in images]
            outputs = model(images)
            for i, (out, tgt) in enumerate(zip(outputs, targets)):
                gt_boxes  = tgt['boxes'].cpu()
                gt_labels = tgt['labels'].cpu()
                pd_boxes  = out['boxes'].cpu()
                pd_scores = out['scores'].cpu()
                pd_labels = out['labels'].cpu()
                img_id    = img_offset + i
                for cls in range(1, NUM_CLASSES):
                    gt_mask = gt_labels == cls
                    pd_mask = pd_labels == cls
                    all_gts[cls].append({'img_id': img_id, 'boxes': gt_boxes[gt_mask],
                                         'matched': torch.zeros(gt_mask.sum(), dtype=torch.bool)})
                    scores = pd_scores[pd_mask]; boxes = pd_boxes[pd_mask]
                    order  = scores.argsort(descending=True)
                    all_preds[cls].append({'img_id': img_id, 'scores': scores[order], 'boxes': boxes[order]})
            img_offset += len(images)

    aps, per_class = [], {}
    for cls in range(1, NUM_CLASSES):
        preds_flat = [(p['img_id'], s.item(), b)
                      for p in all_preds[cls]
                      for s, b in zip(p['scores'], p['boxes'])]
        preds_flat.sort(key=lambda x: -x[1])
        gt_by_img = {g['img_id']: g for g in all_gts[cls]}
        tp  = np.zeros(len(preds_flat))
        fp  = np.zeros(len(preds_flat))
        n_gt = sum(len(g['boxes']) for g in all_gts[cls])
        for k, (img_id, score, box) in enumerate(preds_flat):
            gt = gt_by_img.get(img_id)
            if gt is None or len(gt['boxes']) == 0: fp[k] = 1; continue
            ious = torchvision.ops.box_iou(box.unsqueeze(0), gt['boxes'])[0]
            best_iou, best_j = (ious.max(0) if len(ious) else (torch.tensor(0.), 0))
            if best_iou.item() >= iou_threshold and not gt['matched'][best_j]:
                tp[k] = 1; gt['matched'][best_j] = True
            else: fp[k] = 1
        cum_tp    = np.cumsum(tp); cum_fp = np.cumsum(fp)
        recall    = np.concatenate([[0], cum_tp / max(n_gt, 1), [1]])
        precision = np.concatenate([[1], cum_tp / (cum_tp + cum_fp + 1e-9), [0]])
        for j in range(len(precision) - 2, -1, -1):
            precision[j] = max(precision[j], precision[j + 1])
        ap = float(np.sum((recall[1:] - recall[:-1]) * precision[1:]))
        aps.append(ap); per_class[REV_LABEL_MAP[cls]] = round(ap, 4)
    return float(np.mean(aps)) if aps else 0.0, per_class

print('mAP helper ready')

#  7. Training loop 

In [ ]:

def train_model(model, train_loader, val_loader, epochs=EPOCHS):
    backbone_params = [p for n, p in model.named_parameters() if 'backbone' in n and p.requires_grad]
    head_params     = [p for n, p in model.named_parameters() if 'backbone' not in n and p.requires_grad]
    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': LEARNING_RATE * 0.1},
        {'params': head_params,     'lr': LEARNING_RATE},
    ], weight_decay=WEIGHT_DECAY)
    scheduler = warmup_cosine_schedule(optimizer, 5, epochs, len(train_loader))
    ema      = ModelEMA(model, decay=0.9999)
    best_map = 0.0; history = []

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        loss_comp  = {'cls': 0., 'box': 0., 'rpn_cls': 0., 'rpn_box': 0.}
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1:3d}/{epochs}')
        for images, targets in pbar:
            images  = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
            loss_dict = model(images, targets)
            loss      = sum(loss_dict.values())
            for k, v in loss_dict.items():
                short = k.replace('loss_', '')
                if short in loss_comp: loss_comp[short] += v.item()
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(backbone_params + head_params, max_norm=5.0)
            optimizer.step(); scheduler.step(); ema.update(model)
            epoch_loss += loss.item()
            pbar.set_postfix(loss=f'{loss.item():.3f}', lr=f'{scheduler.get_last_lr()[-1]:.1e}')

        avg_loss = epoch_loss / len(train_loader)
        row = {'epoch': epoch+1, 'loss': avg_loss, 'lr': scheduler.get_last_lr()[-1]}
        map_score = 0.0
        if val_loader and (epoch+1) % 5 == 0:
            map_score, per_cls = compute_map(ema.ema, val_loader)
            model.train(); row['map'] = map_score
            print(f'  per-class AP: {per_cls}')
        history.append(row)
        print(f'  loss={avg_loss:.4f}  cls={loss_comp["cls"]/len(train_loader):.3f}'
              f'  box={loss_comp["box"]/len(train_loader):.3f}'
              f'  lr={scheduler.get_last_lr()[-1]:.2e}'
              + (f'  mAP={map_score:.4f}' if map_score else ''))

        save = (map_score > best_map) if map_score else (avg_loss < getattr(train_model, '_best_loss', 1e9))
        if save:
            best_map = map_score if map_score else best_map
            train_model._best_loss = avg_loss
            save_checkpoint({'model': model.state_dict(), 'epoch': epoch,
                             'best_map': best_map, 'best_loss': avg_loss}, CHECKPOINT_PATH)
            save_checkpoint(ema.ema.state_dict(), EMA_PATH)
            print('  ✓ Saved best checkpoint')

    hist_df = pd.DataFrame(history)
    hist_df.to_csv(OUTPUT_DIR + '/train_history.csv', index=False)
    fig, axes = plt.subplots(1, 2 if 'map' in hist_df.columns else 1, figsize=(14, 4))
    if not isinstance(axes, np.ndarray): axes = [axes]
    axes[0].plot(hist_df['epoch'], hist_df['loss'], marker='o', ms=2)
    axes[0].set(xlabel='Epoch', ylabel='Loss', title='Training Loss'); axes[0].grid(alpha=0.3)
    if 'map' in hist_df.columns:
        mdf = hist_df.dropna(subset=['map'])
        axes[1].plot(mdf['epoch'], mdf['map'], marker='s', ms=4, color='green')
        axes[1].set(xlabel='Epoch', ylabel='mAP@0.5', title='Validation mAP'); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.savefig(OUTPUT_DIR+'/training_curves.png', dpi=120); plt.show()
    return ema.ema

print('train_model ready')

# 8. TTA predict 

In [ ]:

@torch.no_grad()
def tta_predict(model, image_tensor):
    def run(t):
        out = model(t)[0]
        return {k: v.cpu() for k, v in out.items()}

    outputs = [run(image_tensor)]

    img_hf = F.hflip(image_tensor[0]).unsqueeze(0)
    out_hf = run(img_hf); w = image_tensor.shape[-1]
    b_hf = out_hf['boxes'].clone(); b_hf[:, [0, 2]] = w - b_hf[:, [2, 0]]
    outputs.append({'boxes': b_hf, 'scores': out_hf['scores'], 'labels': out_hf['labels']})

    img_vf = F.vflip(image_tensor[0]).unsqueeze(0)
    out_vf = run(img_vf); h = image_tensor.shape[-2]
    b_vf = out_vf['boxes'].clone(); b_vf[:, [1, 3]] = h - b_vf[:, [3, 1]]
    outputs.append({'boxes': b_vf, 'scores': out_vf['scores'], 'labels': out_vf['labels']})

    all_boxes  = torch.cat([o['boxes']  for o in outputs])
    all_scores = torch.cat([o['scores'] for o in outputs])
    all_labels = torch.cat([o['labels'] for o in outputs])
    if len(all_boxes) == 0: return outputs[0]

    fb, fs, fl = [], [], []
    for cls in all_labels.unique():
        mask = all_labels == cls
        keep = nms(all_boxes[mask], all_scores[mask], iou_threshold=0.45)
        fb.append(all_boxes[mask][keep]); fs.append(all_scores[mask][keep]); fl.append(all_labels[mask][keep])
    return {'boxes': torch.cat(fb), 'scores': torch.cat(fs), 'labels': torch.cat(fl)}

print('TTA helper ready')

# 9. Predict & export 

In [ ]:

def predict_and_export(model, val_dir, save_csv_path,
                        save_img_dir=VIZ_DIR, use_tta=True):
    os.makedirs(save_img_dir, exist_ok=True)
    model.eval()
    rows       = []
    val_images = sorted([f for f in os.listdir(val_dir) if f.lower().endswith('.jpg')])
    val_tf     = get_val_transforms()

    for img_file in tqdm(val_images, desc='Predicting'):
        orig_img       = np.array(Image.open(os.path.join(val_dir, img_file)).convert('RGB'))
        orig_h, orig_w = orig_img.shape[:2]

        aug          = val_tf(image=orig_img, bboxes=[], class_labels=[])
        image_tensor = aug['image'].unsqueeze(0).to(DEVICE)

        if use_tta:
            output = tta_predict(model, image_tensor)
        else:
            with torch.no_grad():
                out    = model(image_tensor)[0]
                output = {k: v.cpu() for k, v in out.items()}

        # Scale boxes back to original resolution
        inp_h, inp_w = image_tensor.shape[-2:]
        scale_x = orig_w / inp_w
        scale_y = orig_h / inp_h
        boxes  = output['boxes'].numpy().copy()
        scores = output['scores'].numpy()
        labels = output['labels'].numpy()
        if len(boxes):
            boxes[:, [0, 2]] *= scale_x
            boxes[:, [1, 3]] *= scale_y

        # ── Visualise ────────────────────────────────────────────────────
        fig, ax = plt.subplots(1, figsize=(5, 5))
        ax.imshow(orig_img); ax.axis('off')
        for box, score, label in zip(boxes, scores, labels):
            x1, y1, x2, y2 = map(int, box.tolist())
            lname = REV_LABEL_MAP.get(int(label), 'unknown')
            rect  = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                       linewidth=1.5, edgecolor='red', facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, max(y1-4, 0), f'{lname} {score:.2f}', color='red',
                    fontsize=7, weight='bold',
                    bbox=dict(facecolor='white', alpha=0.5, pad=1, edgecolor='none'))
        fig.tight_layout(pad=0)
        fig.savefig(os.path.join(save_img_dir, img_file), bbox_inches='tight', dpi=100)
        plt.close(fig)

        # ── Build ONE row per image, space-separated multi-detection ─────
        if len(boxes):
            # Sort by score descending — highest confidence first
            order  = np.argsort(scores)[::-1]
            boxes  = boxes[order]
            scores = scores[order]
            labels = labels[order]

            best_label = REV_LABEL_MAP.get(int(labels[0]), 'unknown')

            # Round cf to 2 decimal places (matches sample: 0.10, 0.09 0.05)
            cf_str   = ' '.join(f'{s:.2f}'     for s in scores)
            xmin_str = ' '.join(str(int(b[0])) for b in boxes)
            ymin_str = ' '.join(str(int(b[1])) for b in boxes)
            xmax_str = ' '.join(str(int(b[2])) for b in boxes)
            ymax_str = ' '.join(str(int(b[3])) for b in boxes)

            rows.append({
                'ID':    img_file,
                'label': best_label,
                'cf':    cf_str,
                'xmin':  xmin_str,
                'ymin':  ymin_str,
                'xmax':  xmax_str,
                'ymax':  ymax_str,
            })
        else:
            # No detection — matches sample format: none, 0, 0, 0, 0, 0
            rows.append({
                'ID':    img_file,
                'label': 'none',
                'cf':    '0',
                'xmin':  '0',
                'ymin':  '0',
                'xmax':  '0',
                'ymax':  '0',
            })

    df = pd.DataFrame(rows, columns=['ID', 'label', 'cf', 'xmin', 'ymin', 'xmax', 'ymax'])
    df.to_csv(save_csv_path, index=False)
    n_detected = (df['label'] != 'none').sum()
    print(f'\n✓ submission.csv  → {save_csv_path}')
    print(f'  {len(df)} rows total | {n_detected} with detections | {len(df)-n_detected} none')
    print(f'✓ viz images      → {save_img_dir}/')
    return df

# 10. Build data loaders 

In [ ]:

train_dataset = NEUDETDataset(TRAIN_IMG_ROOT, TRAIN_ANN_ROOT, transforms=get_train_transforms())
sampler       = get_balanced_sampler(train_dataset)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
                            collate_fn=collate_fn, num_workers=2, pin_memory=True)

val_size    = min(200, int(0.1 * len(train_dataset)))
val_indices = list(range(len(train_dataset) - val_size, len(train_dataset)))
val_dataset = torch.utils.data.Subset(
    NEUDETDataset(TRAIN_IMG_ROOT, TRAIN_ANN_ROOT, transforms=get_val_transforms()),
    val_indices)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False,
                         collate_fn=collate_fn, num_workers=2, pin_memory=True)

print(f'train batches: {len(train_loader)} | val batches: {len(val_loader)}')

#  11. Train or load, then infer

In [ ]:

if not INFER_ONLY:
    model = train_model(model, train_loader, val_loader, epochs=EPOCHS)
else:
    wp    = EMA_PATH if os.path.exists(EMA_PATH) else CHECKPOINT_PATH
    state = torch.load(wp, map_location='cpu')
    model.load_state_dict(state['model'] if 'model' in state else state)
    model = model.to(DEVICE)
    print(f'Loaded weights from {wp}')

submission_df = predict_and_export(
    model, VAL_IMG_ROOT, SUBMISSION_PATH,
    save_img_dir=VIZ_DIR, use_tta=USE_TTA)

submission_df.head(10)

# 12. Sanity checks 

In [ ]:

print('Label distribution in submission:')
print(submission_df['label'].value_counts())
print(f'\nTotal rows   : {len(submission_df)}')
print(f'Unique images: {submission_df["ID"].nunique()}')
print('\nSample rows:')
print(submission_df.head(10).to_string(index=False))

imgs = sorted(os.listdir(VIZ_DIR))[:12]
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for ax, fname in zip(axes.flatten(), imgs):
    ax.imshow(Image.open(os.path.join(VIZ_DIR, fname)))
    ax.set_title(fname.split('.')[0], fontsize=7); ax.axis('off')
for ax in axes.flatten()[len(imgs):]: ax.axis('off')
plt.suptitle('Sample predictions', fontsize=13)
plt.tight_layout(); plt.show()

# Print output paths

In [ ]:
import os

paths = {
    "Submission CSV": SUBMISSION_PATH,
    "Train History CSV": OUTPUT_DIR + '/train_history.csv',
    "Best Checkpoint": CHECKPOINT_PATH,
    "EMA Model": EMA_PATH,
    "Viz Images Dir": VIZ_DIR,
}

print("=" * 50)
print("OUTPUT FILE LOCATIONS")
print("=" * 50)
for name, path in paths.items():
    exists = os.path.exists(path)
    status = "✅ EXISTS" if exists else "❌ NOT FOUND"
    size = f"  ({os.path.getsize(path) / 1024:.1f} KB)" if exists and os.path.isfile(path) else ""
    print(f"{status}  |  {name}")
    print(f"          →  {path}{size}")
print("=" * 50)